# GazeSpeakZero — E1b: does a 0.5–1 s dwell remove webcam gaze jitter?

Data: **Eye of the Typer (EOTT)**, Papoutsaki et al., ETRA 2018, Brown HCI group. 51 people, laptop and desktop webcams, with a Tobii Pro X3-120 eye tracker recording where they really looked. It is a free direct download under GPLv3, with no registration.

Run the cells **one at a time** (Shift + Enter), not Run all. CPU is enough. The download cell prints progress every 20 seconds. **Check the dry-run output before letting the long extraction run.** Everything is saved to Google Drive, so if Colab disconnects, run all again and it resumes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/gazespeakzero/E1b'
import os; os.makedirs(WORK, exist_ok=True); print(WORK)
!df -h /content | tail -1

In [ ]:
!pip -q install "mediapipe>=0.10.14" av scikit-learn pandas scipy
import mediapipe as mp, av; print('mediapipe', mp.__version__, '| av', av.__version__)

## 1. Download EOTT (resumable). The zip is never fully unpacked.

In [ ]:
# Download with visible progress, time-outs and resume (safe to re-run)
import os, time, shutil, requests
URL = 'https://webgazer.cs.brown.edu/data/WebGazerETRA2018Dataset_Release20180420.zip'
OUT = '/content/eott.zip'
print('contacting server ...', flush=True)
h = requests.head(URL, timeout=30, allow_redirects=True)
size = int(h.headers.get('content-length', 0))
free = shutil.disk_usage('/content').free
print(f'server status {h.status_code} | file size {size/1e9:.1f} GB | free disk {free/1e9:.1f} GB', flush=True)
assert h.status_code == 200, 'server did not answer normally - stop and note the status code'
assert size == 0 or size < free - 5e9, 'NOT ENOUGH DISK - stop; note both numbers'
have = os.path.getsize(OUT) if os.path.exists(OUT) else 0
if size and have >= size:
    print('already downloaded')
else:
    hdr = {'Range': f'bytes={have}-'} if have else {}
    t0, last, got = time.time(), 0, have
    with requests.get(URL, headers=hdr, stream=True, timeout=(30, 120)) as r, open(OUT, 'ab' if have else 'wb') as f:
        r.raise_for_status()
        for chunk in r.iter_content(8 * 1024 * 1024):
            f.write(chunk); got += len(chunk)
            if time.time() - last > 20:
                last = time.time(); rate = (got - have) / max(last - t0, 1)
                eta = (size - got) / rate / 60 if size and rate else float('nan')
                print(f'{got/1e9:6.2f} / {size/1e9:.2f} GB   {rate/1e6:5.1f} MB/s   about {eta:4.0f} min left', flush=True)
print('done:', os.path.getsize(OUT) / 1e9, 'GB')

## 1b. Look inside the zip (no unpacking; nothing is written to disk)

In [ ]:
# Free the disk from any earlier attempt that unpacked the whole dataset
!rm -rf /content/eott
import zipfile, shutil, re, collections
zf = zipfile.ZipFile('/content/eott.zip')
names = [i for i in zf.infolist() if not i.is_dir()]
parts = collections.Counter(next((q for q in i.filename.split('/')[:-1] if re.fullmatch(r'P_\d+', q)), None) for i in names)
parts.pop(None, None)
need = [i for i in names if re.search(r'-study-(dot_test|dot_test_final|fitts_law)\.webm$', i.filename)]
print('files in zip:', len(names), '| participants:', len(parts))
print('task videos E1b uses:', len(need), '|', round(sum(i.file_size for i in need)/1e9, 2), 'GB in total')
per = collections.Counter()
for i in need:
    per[next(q for q in i.filename.split('/')[:-1] if re.fullmatch(r'P_\d+', q))] += i.file_size
print('largest participant to unpack at one time:', round(max(per.values())/1e6), 'MB')
print('free disk now:', round(shutil.disk_usage('/content').free/1e9, 1), 'GB')
zf.close()
!wget -q -O /content/participant_characteristics.csv https://raw.githubusercontent.com/brownhci/WebGazer/master/www/data/src/participant_characteristics.csv
!echo "characteristics rows: $(grep -c '^P_' /content/participant_characteristics.csv)"


In [ ]:
!mkdir -p /content/e1 /content/e1b
![ -f /content/face_landmarker.task ] || wget -q -O /content/face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

In [ ]:
%%writefile /content/e1/extract_features.py
"""
E1, step 1: run MediaPipe FaceLandmarker on every MPIIFaceGaze image and save
per-frame features to one CSV per subject.

Usage
-----
python extract_features.py --data MPIIFaceGaze --model face_landmarker.task \
    --out features [--max-per-subject 50] [--subjects p00 p01]

Resumable: a subject whose CSV already holds every annotated frame is skipped.
A dry run (--max-per-subject) writes pNN.dry.csv, which analyze.py ignores.

Nothing here is tuned on the data. The features are the raw quantities that
the estimators in analyze.py consume.
"""
from __future__ import annotations

import argparse
import glob
import os
import sys
import time

import cv2
import numpy as np
import pandas as pd
from scipy.io import loadmat

import mediapipe as mp
from mediapipe.tasks.python import BaseOptions
from mediapipe.tasks.python import vision

# MediaPipe 478-point topology. "R" = the subject's right eye (image left).
R_OUT, R_IN, R_UP, R_LO = 33, 133, 159, 145
L_OUT, L_IN, L_UP, L_LO = 263, 362, 386, 374
R_IRIS, L_IRIS = 468, 473          # iris centres
NOSE_TIP = 1
ORIG_IRIS = slice(468, 478)        # exactly what the original repo averages

EYE_BLENDSHAPES = [
    "eyeLookInLeft", "eyeLookInRight", "eyeLookOutLeft", "eyeLookOutRight",
    "eyeLookUpLeft", "eyeLookUpRight", "eyeLookDownLeft", "eyeLookDownRight",
    "eyeBlinkLeft", "eyeBlinkRight", "eyeSquintLeft", "eyeSquintRight",
]


def read_annotations(subject_dir: str) -> pd.DataFrame:
    sid = os.path.basename(subject_dir.rstrip("/\\"))
    txt = os.path.join(subject_dir, f"{sid}.txt")
    rows = []
    with open(txt) as f:
        lines = [ln for ln in f if ln.strip()]
    for ln in lines:
        parts = ln.split()
        if len(parts) != 28:
            raise SystemExit(f"{txt}: expected 28 fields per line, got {len(parts)}: {ln[:80]!r}")
        rows.append(parts)
    cols = (["file", "gt_px_x", "gt_px_y"]
            + [f"ann_lm{i}_{a}" for i in range(6) for a in "xy"]
            + [f"ann_hr{i}" for i in range(3)] + [f"ann_ht{i}" for i in range(3)]
            + [f"ann_fc{i}" for i in range(3)] + [f"ann_gt{i}" for i in range(3)]
            + ["ann_eval_eye"])
    df = pd.DataFrame(rows, columns=cols)
    for c in cols:
        if c not in ("file", "ann_eval_eye"):
            df[c] = df[c].astype(float)
    df.insert(0, "subject", sid)
    return df


def read_screen(subject_dir: str) -> dict:
    m = loadmat(os.path.join(subject_dir, "Calibration", "screenSize.mat"))
    keys = {k.lower(): k for k in m if not k.startswith("__")}

    def g(dim, unit):
        hit = [v for k, v in keys.items() if dim in k and unit in k]
        if len(hit) != 1:
            raise SystemExit(f"screenSize.mat: cannot find {dim}/{unit} among {list(keys.values())}")
        return float(np.squeeze(m[hit[0]]))
    return {"screen_w_px": g("width", "pix"), "screen_h_px": g("height", "pix"),
            "screen_w_mm": g("width", "mm"), "screen_h_mm": g("height", "mm")}


def eye_features(P: np.ndarray, out_i, in_i, up_i, lo_i, iris_i, tag: str) -> dict:
    """Iris position inside the eye opening, in pixel space (aspect-correct)."""
    a, b, iris = P[out_i], P[in_i], P[iris_i]
    axis = b - a
    w = np.linalg.norm(axis) + 1e-9
    u = axis / w
    n = np.array([-u[1], u[0]])
    if n[1] < 0:            # make +v point down the image for both eyes
        n = -n
    d = iris - a
    up, lo = P[up_i], P[lo_i]
    open_h = np.linalg.norm(lo - up) + 1e-9
    denom = (lo - up) @ n
    vlid = float(np.clip(((iris - up) @ n) / denom, -1.0, 2.0)) if open_h / w >= 0.05 and abs(denom) > 1e-6 else float("nan")
    return {
        f"{tag}_h": float(d @ u / w),                 # 0 = outer corner, 1 = inner (mirrored between eyes by design)
        f"{tag}_v": float(d @ n / w),                 # offset below the corner line, same sign for both eyes
        f"{tag}_vlid": vlid,                          # 0 = upper lid, 1 = lower lid; NaN when the eye is nearly shut
        f"{tag}_open": float(open_h / w),
        f"{tag}_width_px": float(w),
    }


def rotation_features(M: np.ndarray) -> dict:
    A = M[:3, :3]
    t = M[:3, 3]
    U, S, Vt = np.linalg.svd(A)          # strip any scale: nearest rotation
    R = U @ Vt
    if np.linalg.det(R) < 0:
        U[:, -1] *= -1
        R = U @ Vt
    # Euler angles (degrees) for R = Rz(roll) @ Ry(yaw) @ Rx(pitch)
    sy = np.clip(-R[2, 0], -1.0, 1.0)
    yaw = np.degrees(np.arcsin(sy))
    pitch = np.degrees(np.arctan2(R[2, 1], R[2, 2]))
    roll = np.degrees(np.arctan2(R[1, 0], R[0, 0]))
    f = {f"R{i}{j}": float(R[i, j]) for i in range(3) for j in range(3)}
    f.update({"tx": float(t[0]), "ty": float(t[1]), "tz": float(t[2]), "R_scale": float(S.mean()),
              "yaw": float(yaw), "pitch": float(pitch), "roll": float(roll)})
    return f


def process_image(landmarker, path: str) -> dict:
    bgr = cv2.imread(path)
    if bgr is None:
        return {"detected": 0, "read_error": 1}
    H, W = bgr.shape[:2]
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    t0 = time.perf_counter()
    res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    dt = (time.perf_counter() - t0) * 1000
    out = {"img_w": W, "img_h": H, "landmark_ms": dt, "read_error": 0}
    out.update(features_from_result(res, W, H))
    return out


def features_from_result(res, W: int, H: int) -> dict:
    """Per-frame features from a FaceLandmarker result (shared by E1 and E1b)."""
    out = {}
    if not res.face_landmarks:
        out["detected"] = 0
        return out
    lms = res.face_landmarks[0]
    out["detected"] = 1
    out["n_landmarks"] = len(lms)
    xy = np.array([[p.x, p.y] for p in lms])           # normalised
    P = xy * np.array([W, H])                           # pixels

    # --- M0: the original repo, verbatim arithmetic ---
    iris = xy[ORIG_IRIS]
    out["m0_px"] = float(iris[:, 0].mean())
    out["m0_py"] = float(iris[:, 1].mean())
    out["m0_cx"] = out["m0_px"] - float(xy[NOSE_TIP, 0]) + 0.5
    out["m0_cy"] = out["m0_py"] - float(xy[NOSE_TIP, 1]) + 0.5

    out.update(eye_features(P, R_OUT, R_IN, R_UP, R_LO, R_IRIS, "reye"))
    out.update(eye_features(P, L_OUT, L_IN, L_UP, L_LO, L_IRIS, "leye"))
    out["nose_x"] = float(xy[NOSE_TIP, 0])
    out["nose_y"] = float(xy[NOSE_TIP, 1])
    out["interocular_norm"] = float(np.linalg.norm(P[R_IRIS] - P[L_IRIS]) / W)
    for k, idx in (("mp_rout", R_OUT), ("mp_rin", R_IN), ("mp_lout", L_OUT), ("mp_lin", L_IN)):
        out[f"{k}_x"], out[f"{k}_y"] = float(P[idx, 0]), float(P[idx, 1])

    if res.face_blendshapes:
        bs = {c.category_name: c.score for c in res.face_blendshapes[0]}
        for k in EYE_BLENDSHAPES:
            out[f"bs_{k}"] = float(bs.get(k, np.nan))
    if res.facial_transformation_matrixes:
        out.update(rotation_features(np.asarray(res.facial_transformation_matrixes[0])))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="path to the unzipped MPIIFaceGaze folder")
    ap.add_argument("--model", required=True, help="face_landmarker.task")
    ap.add_argument("--out", default="features")
    ap.add_argument("--subjects", nargs="*", default=None)
    ap.add_argument("--max-per-subject", type=int, default=None,
                    help="dry run: first N images of each subject")
    args = ap.parse_args()

    os.makedirs(args.out, exist_ok=True)
    subj_dirs = sorted(d for d in glob.glob(os.path.join(args.data, "p[0-9][0-9]")) if os.path.isdir(d))
    if args.subjects:
        subj_dirs = [d for d in subj_dirs if os.path.basename(d) in args.subjects]
    if not subj_dirs:
        sys.exit(f"No pNN folders found under {args.data}")

    opts = vision.FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=args.model),
        running_mode=vision.RunningMode.IMAGE,
        num_faces=1,
        output_face_blendshapes=True,
        output_facial_transformation_matrixes=True,
    )
    with vision.FaceLandmarker.create_from_options(opts) as lm:
        for sd in subj_dirs:
            sid = os.path.basename(sd)
            dry = args.max_per_subject is not None
            dest = os.path.join(args.out, f"{sid}.dry.csv" if dry else f"{sid}.csv")
            ann = read_annotations(sd)
            if not dry and os.path.exists(dest):
                n_have = len(pd.read_csv(dest, usecols=["file"]))
                if n_have == len(ann):
                    print(f"{sid}: complete ({n_have} frames), skipping")
                    continue
                print(f"{sid}: incomplete ({n_have}/{len(ann)}), redoing")
            if dry:
                ann = ann.iloc[:args.max_per_subject]
            scr = read_screen(sd)
            recs = []
            t0 = time.time()
            for i, row in enumerate(ann.itertuples(index=False)):
                r = process_image(lm, os.path.join(sd, row.file))
                recs.append(r)
                if (i + 1) % 500 == 0:
                    print(f"  {sid} {i+1}/{len(ann)}  {time.time()-t0:.0f}s", flush=True)
            feat = pd.DataFrame(recs)
            df = pd.concat([ann.reset_index(drop=True), feat], axis=1)
            for k, v in scr.items():
                df[k] = v
            df.to_csv(dest + ".tmp", index=False)
            os.replace(dest + ".tmp", dest)
            print(f"{sid}: {len(df)} frames, detected {df['detected'].mean():.3f}, "
                  f"unreadable {int(df['read_error'].sum())}, {time.time()-t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/e1/analyze.py
"""
E1, step 2: calibration-free gaze-zone accuracy on MPIIFaceGaze.

Reads the per-subject CSVs from extract_features.py and writes tables,
figures and a markdown report to --out.

    python analyze.py --features features --out results

Protocol
--------
* Ground truth: the on-screen target (pixels) divided by that subject's
  screen size gives a position in [0, 1]^2; the zone is its grid cell.
* Every learned method is leave-one-subject-out (LOSO): trained on 14 people,
  tested on the 15th, so no calibration data from the test person is used.
* A frame with no detected face counts as WRONG in every accuracy figure.
* Headline = mean of the 15 per-subject accuracies, with a 95 % bootstrap CI
  over subjects. The pooled frame-level figure is reported alongside.
* Calibration reference: k frames of the test person correct the M2 output
  (k < 5: offset only; k >= 5: per-axis least-squares slope and offset,
  slope clipped to [0.5, 3]). Two draws of the k frames:
    - "first session": from the person's first recording day, scored on
      the other days (how a real calibration is used);
    - "random": from any day, scored on the rest.
  20 draws per k.
"""
from __future__ import annotations

import argparse
import glob
import json
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

GRIDS = [(1, 2), (2, 1), (1, 3), (2, 2), (3, 3), (4, 4)]
CALIB_K = [1, 3, 5, 9, 16]
N_CALIB_DRAWS = 20
N_BOOT = 10000
SEED = 0

EYE = []
for e in ("reye", "leye"):
    EYE += [f"{e}_h", f"{e}_v", f"{e}_vlid", f"{e}_open"]
BS = ["bs_eyeLookInLeft", "bs_eyeLookInRight", "bs_eyeLookOutLeft", "bs_eyeLookOutRight",
      "bs_eyeLookUpLeft", "bs_eyeLookUpRight", "bs_eyeLookDownLeft", "bs_eyeLookDownRight",
      "bs_eyeBlinkLeft", "bs_eyeBlinkRight", "bs_eyeSquintLeft", "bs_eyeSquintRight"]
HEAD = ([f"R{i}{j}" for i in range(3) for j in range(3)]
        + ["tx", "ty", "tz", "yaw", "pitch", "roll", "nose_x", "nose_y", "interocular_norm"])
FEATURE_SETS = {
    "M0-fit": ["m0_cx", "m0_cy"],
    "M1": EYE + BS,
    "M2": EYE + BS + HEAD,
}


# ----------------------------------------------------------------------------
def load(features_dir: str) -> pd.DataFrame:
    files = sorted(glob.glob(os.path.join(features_dir, "p[0-9][0-9].csv")))
    if not files:
        raise SystemExit(f"no pNN.csv in {features_dir}")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    counts = df.groupby("subject").size()
    print("frames per subject:", counts.to_dict(), flush=True)
    if counts.min() < 0.9 * counts.max():
        print("WARNING: some subjects have far fewer frames than others", flush=True)
    df["day"] = df["file"].str.split("/").str[0]
    df["gx"] = df["gt_px_x"] / df["screen_w_px"]
    df["gy"] = df["gt_px_y"] / df["screen_h_px"]
    df["detected"] = df["detected"].fillna(0).astype(int)
    return df


def zone_of(x, y, rows, cols):
    """Grid cell of normalised screen positions; NaN in -> -1 (always wrong)."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    bad = ~(np.isfinite(x) & np.isfinite(y))
    c = np.clip(np.floor(np.nan_to_num(x) * cols), 0, cols - 1).astype(int)
    r = np.clip(np.floor(np.nan_to_num(y) * rows), 0, rows - 1).astype(int)
    z = r * cols + c
    z[bad] = -1
    return z


def m0_original_phi(cx, cy):
    """Repo's _phi for (3,3) with theta = 0.333, verbatim."""
    cx = np.asarray(cx, float)
    cy = np.asarray(cy, float)
    bad = ~(np.isfinite(cx) & np.isfinite(cy))
    col = np.clip(np.floor(np.nan_to_num(cx) / 0.333), 0, 2).astype(int)
    row = np.clip(np.floor(np.nan_to_num(cy) / 0.333), 0, 2).astype(int)
    z = row * 3 + col
    z[bad] = -1
    return z


def make_model(kind: str):
    if kind == "hgb":
        return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                                             early_stopping=False, random_state=SEED)
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=1.0))


def loso_predict(df: pd.DataFrame, feats: list[str], kind: str) -> np.ndarray:
    """Returns an (N, 2) array of predicted (x, y); NaN where no face."""
    pred = np.full((len(df), 2), np.nan)
    det = df["detected"].values == 1
    for s in df["subject"].unique():
        tr = (df["subject"].values != s) & det
        te = (df["subject"].values == s) & det
        if te.sum() == 0:
            continue
        X_tr, X_te = df.loc[tr, feats].values, df.loc[te, feats].values
        for j, tgt in enumerate(("gx", "gy")):
            m = make_model(kind)
            m.fit(X_tr, df.loc[tr, tgt].values)
            pred[te, j] = m.predict(X_te)
    return pred


def per_subject_acc(df, zpred, rows, cols) -> pd.Series:
    zt = zone_of(df["gx"], df["gy"], rows, cols)
    ok = pd.Series(zpred == zt, index=df.index)
    return ok.groupby(df["subject"]).mean()


def boot_ci(v: np.ndarray, rng) -> tuple[float, float]:
    """BCa bootstrap CI of the mean over subjects; percentile if BCa is undefined."""
    from scipy.stats import bootstrap
    v = np.asarray(v, float)
    if np.allclose(v, v[0]):
        return float(v[0]), float(v[0])
    for method in ("BCa", "percentile"):
        r = bootstrap((v,), np.mean, n_resamples=N_BOOT, method=method, random_state=rng)
        lo, hi = r.confidence_interval
        if np.isfinite(lo) and np.isfinite(hi):
            return float(lo), float(hi)
    return float("nan"), float("nan")


def balanced_acc(zt, zp, K) -> float:
    return float(np.mean([np.mean(zp[zt == k] == k) for k in range(K) if np.any(zt == k)]))


def mm_error(df, pred) -> pd.Series:
    dx = (pred[:, 0] - df["gx"].values) * df["screen_w_mm"].values
    dy = (pred[:, 1] - df["gy"].values) * df["screen_h_mm"].values
    return pd.Series(np.sqrt(dx ** 2 + dy ** 2), index=df.index)


def fit_correction(p, t, k):
    if k < 5:
        return 1.0, float(np.mean(t - p))
    pm, tm = p.mean(), t.mean()
    sxx = np.sum((p - pm) ** 2)
    a = float(np.clip(np.sum((p - pm) * (t - tm)) / sxx, 0.5, 3.0)) if sxx > 1e-9 else 1.0
    return a, float(tm - a * pm)


def calibrate(df, base_pred, rows_cols, rng) -> list[dict]:
    """Per-subject k-shot correction of base_pred, scored on frames not used to calibrate."""
    out = []
    det = df["detected"].values == 1
    days = df["day"].values
    for s in df["subject"].unique():
        idx_all = np.where(df["subject"].values == s)[0]
        first_day = sorted(set(days[idx_all]))[0]
        pools = {
            "random": (idx_all[det[idx_all]], None),
            "first session": (idx_all[det[idx_all] & (days[idx_all] == first_day)],
                              idx_all[days[idx_all] != first_day]),
        }
        for scheme, (pool, test_fixed) in pools.items():
            for k in CALIB_K:
                if len(pool) < k or (test_fixed is not None and len(test_fixed) == 0):
                    continue
                for d in range(N_CALIB_DRAWS):
                    cal = rng.choice(pool, size=k, replace=False)
                    test = np.setdiff1d(idx_all, cal) if test_fixed is None else test_fixed
                    corr = np.empty((len(test), 2))
                    for j, tgt in enumerate(("gx", "gy")):
                        a, b = fit_correction(base_pred[cal, j], df[tgt].values[cal], k)
                        corr[:, j] = a * base_pred[test, j] + b   # NaN (no face) stays NaN -> wrong
                    sub = df.iloc[test]
                    for (r, c) in rows_cols:
                        acc = float(np.mean(zone_of(corr[:, 0], corr[:, 1], r, c)
                                            == zone_of(sub["gx"], sub["gy"], r, c)))
                        out.append({"scheme": scheme, "subject": s, "k": k, "draw": d,
                                    "grid": f"{r}x{c}", "acc": acc})
    return out


def confusion_png(df, zpred, rows, cols, title, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    K = rows * cols
    zt = zone_of(df["gx"], df["gy"], rows, cols)
    M = np.zeros((K, K + 1))
    for t, p in zip(zt, zpred):
        M[t, p if p >= 0 else K] += 1
    M = M / M.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(0.55 * K + 2.5, 0.55 * K + 1.8))
    im = ax.imshow(M, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(K + 1), [str(i) for i in range(K)] + ["no face"], rotation=45)
    ax.set_yticks(range(K))
    ax.set_xlabel("predicted zone")
    ax.set_ylabel("true zone")
    ax.set_title(title, fontsize=9)
    for i in range(K):
        for j in range(K + 1):
            if M[i, j] >= 0.005:
                ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if M[i, j] > 0.5 else "black")
    fig.colorbar(im, ax=ax, fraction=0.04)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


# ----------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--features", default="features")
    ap.add_argument("--out", default="results")
    ap.add_argument("--skip-calibration", action="store_true")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    rng = np.random.default_rng(SEED)

    df = load(args.features)
    subjects = sorted(df["subject"].unique())
    det = df["detected"].values == 1

    # ---- data QA -------------------------------------------------------------
    qa = {
        "n_subjects": len(subjects),
        "n_frames": int(len(df)),
        "detection_rate": float(det.mean()),
        "detection_rate_min_subject": float(df.groupby("subject")["detected"].mean().min()),
        "gt_outside_screen_frac": float(((df.gx < 0) | (df.gx >= 1) | (df.gy < 0) | (df.gy >= 1)).mean()),
        "median_landmark_ms": float(df["landmark_ms"].median()),
        "unreadable_images": int(df["read_error"].fillna(0).sum()),
        "frames_per_subject": {k: int(v) for k, v in df.groupby("subject").size().items()},
        "median_R_scale": float(df.loc[det, "R_scale"].median()) if "R_scale" in df else None,
        "median_reye_h": float(df.loc[det, "reye_h"].median()),
    }
    # MediaPipe eye corners vs the dataset's hand-annotated eye corners (first 4 landmarks)
    if det.any():
        d = df[det]
        mp_c = np.stack([d[[f"{k}_x", f"{k}_y"]].values for k in ("mp_rout", "mp_rin", "mp_lout", "mp_lin")], 1)
        an_c = np.stack([d[[f"ann_lm{i}_x", f"ann_lm{i}_y"]].values for i in range(4)], 1)
        dist = np.linalg.norm(an_c[:, :, None, :] - mp_c[:, None, :, :], axis=-1).min(axis=2).mean(axis=1)
        io = d["interocular_norm"].values * d["img_w"].values
        rel = dist / np.maximum(io, 1e-6)
        qa["eye_corner_err_median_rel_interocular"] = float(np.median(rel))
        qa["eye_corner_err_gt_0.2_frac"] = float(np.mean(rel > 0.2))

    # ---- M0 diagnostics: where does the original feature actually land? ---------
    m0_zone = m0_original_phi(df["m0_cx"], df["m0_cy"])
    m0_hist = pd.Series(m0_zone).value_counts(normalize=True).sort_index()
    qa["m0_zone_distribution"] = {int(k): round(float(v), 4) for k, v in m0_hist.items()}
    qa["m0_cx_quantiles"] = df.loc[det, "m0_cx"].quantile([0.01, 0.5, 0.99]).round(4).tolist()
    qa["m0_cy_quantiles"] = df.loc[det, "m0_cy"].quantile([0.01, 0.5, 0.99]).round(4).tolist()
    true33 = zone_of(df["gx"], df["gy"], 3, 3)
    qa["gt_zone_distribution_3x3"] = {int(k): round(float(v), 4)
                                      for k, v in pd.Series(true33).value_counts(normalize=True).sort_index().items()}

    # ---- predictions -----------------------------------------------------------
    preds = {}
    preds["M0 original"] = np.c_[df["m0_cx"], df["m0_cy"]]
    preds["M0 original (x flipped)"] = np.c_[1 - df["m0_cx"], df["m0_cy"]]
    for name, feats in FEATURE_SETS.items():
        for kind in ("ridge", "hgb"):
            print(f"LOSO {name} {kind} ...", flush=True)
            preds[f"{name} {kind}"] = loso_predict(df, feats, kind)

    # majority zone of the training subjects, per grid (LOSO)
    rows_out, per_subj_rows = [], []
    for (r, c) in GRIDS:
        g = f"{r}x{c}"
        zt = zone_of(df["gx"], df["gy"], r, c)
        maj = np.full(len(df), -1)
        for s in subjects:
            tr = df["subject"].values != s
            maj[~tr] = pd.Series(zt[tr]).mode().iloc[0]
        maj = np.where(det, maj, -1)                       # needs a face like every other method
        method_z = {"chance (1/K)": None, "majority zone": maj}
        for name, p in preds.items():
            if name.startswith("M0 original") and (r, c) == (3, 3):
                z = m0_original_phi(p[:, 0], p[:, 1])
            else:
                z = zone_of(p[:, 0], p[:, 1], r, c)
            if name.startswith("M0 original"):
                z = np.where(det, z, -1)
                if (r, c) != (3, 3):
                    name = name.replace("M0 original", "M0 generalised")
            method_z[name] = z
        for name, z in method_z.items():
            if z is None:
                rows_out.append({"grid": g, "method": name, "acc_mean_subj": 1 / (r * c),
                                 "ci_lo": np.nan, "ci_hi": np.nan, "acc_pooled": 1 / (r * c),
                                 "acc_detected_only": 1 / (r * c), "balanced_acc": 1 / (r * c),
                                 "acc_min_subj": np.nan, "acc_max_subj": np.nan})
                continue
            ps = per_subject_acc(df, z, r, c)
            lo, hi = boot_ci(ps.values, rng)
            rows_out.append({"grid": g, "method": name, "acc_mean_subj": ps.mean(),
                             "ci_lo": lo, "ci_hi": hi, "acc_pooled": float(np.mean(z == zt)),
                             "acc_detected_only": float(np.mean(z[det] == zt[det])),
                             "balanced_acc": balanced_acc(zt, z, r * c),
                             "acc_min_subj": ps.min(), "acc_max_subj": ps.max()})
            for s, a in ps.items():
                per_subj_rows.append({"grid": g, "method": name, "subject": s, "acc": a})
        if (r, c) in [(3, 3), (2, 2), (1, 3)]:
            m0 = "M0 original" if (r, c) == (3, 3) else "M0 generalised"
            for name, fname in ((m0, "M0"), ("M2 hgb", "M2_hgb")):
                confusion_png(df, method_z[name], r, c, f"{name}, {g} (rows: true, cols: predicted)",
                              os.path.join(args.out, f"confusion_{fname}_{g}.png"))

    acc = pd.DataFrame(rows_out)
    acc.to_csv(os.path.join(args.out, "e1_zone_accuracy.csv"), index=False)
    pd.DataFrame(per_subj_rows).to_csv(os.path.join(args.out, "e1_per_subject.csv"), index=False)

    mm = []
    for name, p in preds.items():
        if name.startswith("M0"):
            continue
        e = mm_error(df, p)[det]
        mm.append({"method": name, "median_mm": e.median(), "mean_mm": e.mean(),
                   "p90_mm": e.quantile(0.9)})
    mm = pd.DataFrame(mm)
    mm.to_csv(os.path.join(args.out, "e1_mm_error.csv"), index=False)

    cal = None
    if not args.skip_calibration:
        print("calibration reference ...", flush=True)
        cal = pd.DataFrame(calibrate(df, preds["M2 hgb"], [(1, 3), (2, 2), (3, 3), (4, 4)], rng))
        cal.to_csv(os.path.join(args.out, "e1_calibration_raw.csv"), index=False)
        cal = (cal.groupby(["scheme", "grid", "k", "subject"])["acc"].mean()
                  .groupby(["scheme", "grid", "k"]).agg(["mean", "min", "max"]).reset_index())
        cal.to_csv(os.path.join(args.out, "e1_calibration.csv"), index=False)

    with open(os.path.join(args.out, "e1_qa.json"), "w") as f:
        json.dump(qa, f, indent=2)

    # ---- report ------------------------------------------------------------------
    L = ["# E1: calibration-free gaze-zone accuracy on MPIIFaceGaze", "",
         f"{qa['n_subjects']} subjects, {qa['n_frames']} frames. Face detected in "
         f"{qa['detection_rate']:.1%} of frames (lowest subject {qa['detection_rate_min_subject']:.1%}); "
         "undetected frames count as wrong.", "",
         "Accuracy = mean of per-subject accuracies (95% bootstrap CI over subjects). "
         "All learned methods are leave-one-subject-out.", ""]
    for g in [f"{r}x{c}" for r, c in GRIDS]:
        L += [f"## Grid {g}", "",
              "| Method | Accuracy | 95% CI | Pooled | Face-found frames only | Balanced | Worst subject | Best subject |",
              "|---|---|---|---|---|---|---|---|"]
        for _, x in acc[acc.grid == g].iterrows():
            ci = "" if np.isnan(x.ci_lo) else f"{x.ci_lo:.3f}–{x.ci_hi:.3f}"
            mn = "" if np.isnan(x.acc_min_subj) else f"{x.acc_min_subj:.3f}"
            mx = "" if np.isnan(x.acc_max_subj) else f"{x.acc_max_subj:.3f}"
            L.append(f"| {x.method} | {x.acc_mean_subj:.3f} | {ci} | {x.acc_pooled:.3f} | "
                     f"{x.acc_detected_only:.3f} | {x.balanced_acc:.3f} | {mn} | {mx} |")
        L.append("")
    L += ["## On-screen error (detected frames, mm)", "", "| Method | Median | Mean | 90th pct |",
          "|---|---|---|---|"]
    for _, x in mm.iterrows():
        L.append(f"| {x.method} | {x.median_mm:.0f} | {x.mean_mm:.0f} | {x.p90_mm:.0f} |")
    if cal is not None:
        L += ["", "## With k calibration frames from the test person (M2 hgb + correction)", "",
              "k < 5: offset only; k >= 5: slope and offset per axis. "
              "'first session' calibrates on day 1 and scores the other days.", "",
              "| Scheme | Grid | k | Mean acc | Worst subject | Best subject |", "|---|---|---|---|---|---|"]
        for _, x in cal.iterrows():
            L.append(f"| {x.scheme} | {x.grid} | {x.k} | {x['mean']:.3f} | {x['min']:.3f} | {x['max']:.3f} |")
    L += ["", "## Data QA and M0 diagnostics", "", "```", json.dumps(qa, indent=2), "```"]
    with open(os.path.join(args.out, "E1_REPORT.md"), "w") as f:
        f.write("\n".join(L) + "\n")
    print("\n".join(L))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/e1b/eott_extract.py
"""
E1b, step 1: per-frame webcam features + synchronised Tobii gaze from the
Eye of the Typer (EOTT) dataset (Papoutsaki et al., ETRA 2018; Brown HCI, GPLv3).

For each participant and each selected task video:
  * decode the webcam .webm with PyAV and time-stamp every frame
    (recording-start epoch from the browser log + frame presentation time);
  * run MediaPipe FaceLandmarker in VIDEO mode and compute exactly the E1
    features (e1/extract_features.features_from_result);
  * attach the Tobii Pro X3-120 gaze sample nearest in time.

Also writes the raw Tobii stream for each task interval, which analysis uses
to find fixations.

    python eott_extract.py --zip eott.zip --chars participant_characteristics.csv \
        --model face_landmarker.task --out eott_features [--participants P_01 P_02] \
        [--max-seconds 20]            # dry run: first 20 s of each video

--zip reads the dataset straight from the zip and unpacks one participant at a
time into a scratch folder that is deleted afterwards, so the disk never holds
more than the zip plus one participant. (--data works on an unpacked folder.)

Resumable per participant (skips a participant whose done-marker exists).

Format facts are taken from the dataset's own extractor
(github.com/brownhci/WebGazer, www/data/src/participant.py and
webgazerExtractServer.py): browser log <startTimestamp>.json with
"recording start" events (sessionString -> '/' replaced by '-' + '.webm',
epoch in ms); Tobii log P_X/P_X.txt, one JSON object per line with
true_time (s), {left,right}_gaze_point_on_display_area (normalised),
{left,right}_pupil_validity.
"""
from __future__ import annotations

import argparse
import csv
import glob
import json
import os
import re
import sys
import time

import numpy as np
import pandas as pd

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.join(HERE, "..", "e1"))
sys.path.insert(0, HERE)

TASK_RE = re.compile(r"-study-(dot_test|dot_test_final|fitts_law)\.webm$")
TOBII_MATCH_MS = 20          # a frame gets a Tobii label only if a sample lies within this


# ----------------------------------------------------------------------------- inputs
def read_characteristics(path: str) -> dict:
    """participant -> {'setup': 'Laptop'|'PC', 'screen_w_px', 'screen_h_px'} (columns as in the extractor)."""
    out = {}
    with open(path, newline="") as f:
        for row in csv.reader(f):
            if not row or not row[0].startswith("P_"):
                continue
            try:
                out[row[0]] = {"setup": row[3].strip(), "screen_w_px": int(row[4]), "screen_h_px": int(row[5])}
            except (IndexError, ValueError):
                continue
    return out


def read_log(pdir: str) -> tuple[list[dict], str]:
    webm = glob.glob(os.path.join(pdir, "*dot_test_instructions.webm"))
    if not webm:
        raise FileNotFoundError(f"{pdir}: no *dot_test_instructions.webm (cannot find the log start timestamp)")
    start = os.path.basename(webm[0]).split("_")[0]
    log_path = os.path.join(pdir, f"{start}.json")
    with open(log_path) as f:
        log = json.load(f)
    return log, log_path


def recording_starts(log: list[dict]) -> dict:
    """video filename -> recording-start epoch (ms)."""
    starts = {}
    for ev in log:
        if ev.get("type") == "recording start" and ev.get("sessionString"):
            fn = ev["sessionString"].replace("/", "-") + ".webm"
            starts[fn] = int(ev["epoch"])
    return starts


def read_tobii(pdir: str) -> pd.DataFrame:
    pid = os.path.basename(pdir.rstrip("/\\"))
    rows = []
    with open(os.path.join(pdir, f"{pid}.txt")) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                d = json.loads(line)
            except json.JSONDecodeError:
                continue                      # the extractor notes one truncated last line
            rv, lv = int(d["right_pupil_validity"]), int(d["left_pupil_validity"])
            def pt(key, valid):
                try:
                    x, y = (float(v) for v in d[key][:2])
                except (TypeError, ValueError, KeyError):
                    return np.nan, np.nan
                return (x, y) if valid and np.isfinite(x) and np.isfinite(y) else (np.nan, np.nan)
            r_ = pt("right_gaze_point_on_display_area", rv)
            l_ = pt("left_gaze_point_on_display_area", lv)
            with np.errstate(all="ignore"):
                x = np.nanmean([r_[0], l_[0]]) if np.isfinite([r_[0], l_[0]]).any() else np.nan
                y = np.nanmean([r_[1], l_[1]]) if np.isfinite([r_[1], l_[1]]).any() else np.nan
            ok = int(np.isfinite(x) and np.isfinite(y))
            rows.append((round(float(d["true_time"]) * 1000), float(x), float(y), ok))
    t = pd.DataFrame(rows, columns=["t_ms", "tx", "ty", "tvalid"]).sort_values("t_ms")
    return t.drop_duplicates("t_ms").reset_index(drop=True)


# ----------------------------------------------------------------------------- video
def frame_times(pts_ms: list) -> np.ndarray:
    """Per-frame times (ms from recording start) from the raw presentation times,
    as the dataset's own extractor does. The dataset notes that many webm frames
    repeat the previous frame's timestamp and some are missing: a stamp that is
    missing or not later than the previous good one is filled by linear
    interpolation between its good neighbours (by frame index). A single line
    fit is NOT used, because webcam frame rates drift and stall."""
    n = len(pts_ms)
    idx = np.arange(n, dtype=float)
    p = np.array([np.nan if v is None else float(v) for v in pts_ms])
    good = np.zeros(n, bool)
    last = -np.inf
    for i in range(n):
        if np.isfinite(p[i]) and p[i] > last:
            good[i] = True
            last = p[i]
    if good.sum() < 5:
        return idx * 33.3
    t = np.interp(idx, idx[good], p[good])
    # extrapolate beyond the last/first good stamp with the median frame period
    per = np.median(np.diff(p[good])) / max(np.median(np.diff(idx[good])), 1)
    lo, hi = np.argmax(good), n - 1 - np.argmax(good[::-1])
    t[:lo] = p[lo] - per * (lo - idx[:lo])
    t[hi + 1:] = p[hi] + per * (idx[hi + 1:] - hi)
    return t


def packet_pts(path: str, max_seconds: float | None) -> list:
    """Pass 1: presentation times (ms) from the container, without decoding pixels.
    VP8/VP9 webcam streams have no B-frames, so packet order is frame order."""
    import av
    pts = []
    with av.open(path) as c:
        s = c.streams.video[0]
        tb = float(s.time_base) if s.time_base else 0.001
        for pkt in c.demux(s):
            if pkt.size == 0:
                continue
            pts.append(None if pkt.pts is None else pkt.pts * tb * 1000.0)
            if max_seconds and pts[-1] is not None and pts[-1] > max_seconds * 1000:
                break
    return pts


def iter_frames(path: str, n_max: int):
    """Pass 2: decoded RGB frames, streamed (a whole video does not fit in memory)."""
    import av
    with av.open(path) as c:
        s = c.streams.video[0]
        s.thread_type = "AUTO"
        for i, fr in enumerate(c.decode(s)):
            if i >= n_max:
                break
            yield fr.to_ndarray(format="rgb24")


# ----------------------------------------------------------------------------- main
def process_participant(pdir, chars, model_path, out_dir, max_seconds):
    import mediapipe as mp
    from mediapipe.tasks.python import BaseOptions, vision
    from extract_features import features_from_result

    pid = os.path.basename(pdir.rstrip("/\\"))
    info = chars.get(pid)
    if info is None:
        print(f"{pid}: not in participant_characteristics.csv, skipped")
        return
    log, _ = read_log(pdir)
    starts = recording_starts(log)
    tob = read_tobii(pdir)
    vids = sorted(v for v in glob.glob(os.path.join(pdir, "*.webm")) if TASK_RE.search(os.path.basename(v)))
    rows, tob_keep = [], []
    vids.sort(key=lambda v: starts.get(os.path.basename(v), float("inf")))   # chronological
    for v in vids:
        n0, k0 = len(rows), len(tob_keep)
        try:
            _one_video(v, pid, info, starts, tob, model_path, max_seconds, rows, tob_keep)
        except Exception as e:
            del rows[n0:], tob_keep[k0:]                  # drop a half-processed video
            print(f"  {pid} {os.path.basename(v)}: FAILED {type(e).__name__}: {e}", flush=True)
    if not rows:
        print(f"{pid}: no task videos processed")
        return
    suffix = ".dry" if max_seconds else ""
    pd.DataFrame(rows).to_csv(os.path.join(out_dir, f"{pid}_frames{suffix}.csv"), index=False)
    pd.concat(tob_keep).to_csv(os.path.join(out_dir, f"{pid}_tobii{suffix}.csv"), index=False)
    if not max_seconds:
        open(os.path.join(out_dir, f"{pid}.done"), "w").close()


def _one_video(v, pid, info, starts, tob, model_path, max_seconds, rows, tob_keep):
    import mediapipe as mp
    from mediapipe.tasks.python import BaseOptions, vision
    from extract_features import features_from_result
    if True:
        fn = os.path.basename(v)
        task = TASK_RE.search(fn).group(1)
        if fn not in starts:
            print(f"  {pid} {fn}: no 'recording start' event, skipped")
            return
        t0 = time.time()
        pts = packet_pts(v, max_seconds)
        if len(pts) < 10:
            print(f"  {pid} {fn}: only {len(pts)} frames, skipped")
            return
        rel = frame_times(pts)
        epoch = starts[fn] + rel
        opts = vision.FaceLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=model_path),
            running_mode=vision.RunningMode.VIDEO, num_faces=1,
            output_face_blendshapes=True, output_facial_transformation_matrixes=True)
        # nearest Tobii sample
        tt = tob["t_ms"].values
        j = np.clip(np.searchsorted(tt, epoch), 1, len(tt) - 1)
        jn = np.where(np.abs(tt[j - 1] - epoch) <= np.abs(tt[j] - epoch), j - 1, j)
        dt = np.abs(tt[jn] - epoch)
        last_ts = -1
        n_done = 0
        with vision.FaceLandmarker.create_from_options(opts) as lm:
            for i, rgb in enumerate(iter_frames(v, len(pts))):
                n_done += 1
                ts = max(int(round(rel[i])), last_ts + 1)
                last_ts = ts
                H, W = rgb.shape[:2]
                res = lm.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb)), ts)
                r = {"participant": pid, "setup": info["setup"], "task": task, "video": fn,
                     "frame": i, "pts_raw_ms": pts[i], "t_rel_ms": float(rel[i]), "epoch_ms": float(epoch[i]),
                     "img_w": W, "img_h": H}
                r.update(features_from_result(res, W, H))
                ok = dt[i] <= TOBII_MATCH_MS and tob["tvalid"].values[jn[i]] == 1
                r["tobii_x"] = float(tob["tx"].values[jn[i]]) if ok else np.nan
                r["tobii_y"] = float(tob["ty"].values[jn[i]]) if ok else np.nan
                r["tobii_dt_ms"] = float(dt[i])
                rows.append(r)
        # raw Tobii for this task's interval (for fixation detection)
        seg = tob[(tob.t_ms >= epoch[0] - 500) & (tob.t_ms <= epoch[-1] + 500)].copy()
        seg.insert(0, "video", fn)
        seg.insert(0, "task", task)
        seg.insert(0, "participant", pid)
        tob_keep.append(seg)
        if n_done != len(pts):
            print(f"  {pid} {task}: WARNING decoded {n_done} frames but container lists {len(pts)}", flush=True)
        fps = (len(pts) - 1) / max((rel[-1] - rel[0]) / 1000, 1e-6)
        gap = float(np.max(np.diff(rel))) if len(rel) > 1 else 0.0
        print(f"  {pid} {task}: {n_done} frames, {fps:.1f} fps, "
              f"{(rel[-1]-rel[0])/1000:.0f} s, longest frame gap {gap:.0f} ms, {time.time()-t0:.0f} s", flush=True)


NEEDED = re.compile(r"(-study-(dot_test|dot_test_final|fitts_law)\.webm|\.json|/P_\d+\.txt)$")


def zip_participants(zf) -> dict:
    """participant id -> list of zip members that E1b needs (read from the zip index only)."""
    groups = {}
    for zi in zf.infolist():
        name = zi.filename
        if zi.is_dir() or name.startswith("__MACOSX") or "/._" in name:
            continue
        parts = name.split("/")
        pid = next((q for q in parts[:-1] if re.fullmatch(r"P_\d+", q)), None)
        if pid is None:
            continue
        g = groups.setdefault(pid, {"need": [], "instr": None})
        if name.endswith("dot_test_instructions.webm"):
            g["instr"] = os.path.basename(name)      # only its NAME is needed (log timestamp)
        elif NEEDED.search("/" + name):
            g["need"].append(zi)
    return groups


def main():
    ap = argparse.ArgumentParser()
    src = ap.add_mutually_exclusive_group(required=True)
    src.add_argument("--data", help="folder that contains the P_* participant folders")
    src.add_argument("--zip", help="the EOTT zip; participants are unpacked one at a time and deleted after use")
    ap.add_argument("--chars", required=True, help="participant_characteristics.csv")
    ap.add_argument("--model", required=True)
    ap.add_argument("--out", default="eott_features")
    ap.add_argument("--tmp", default="/tmp/eott_one", help="scratch folder for --zip mode")
    ap.add_argument("--participants", nargs="*")
    ap.add_argument("--max-seconds", type=float, default=None)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    chars = read_characteristics(a.chars)
    key = lambda pid: int(re.sub(r"\D", "", pid) or 0)

    if a.data:
        pdirs = sorted((d for d in glob.glob(os.path.join(a.data, "P_*")) if os.path.isdir(d)),
                       key=lambda d: key(os.path.basename(d)))
        if a.participants:
            pdirs = [d for d in pdirs if os.path.basename(d) in a.participants]
        if not pdirs:
            sys.exit(f"no P_* folders under {a.data}")
        print(f"{len(pdirs)} participant folders; {len(chars)} in characteristics file", flush=True)
        for d in pdirs:
            pid = os.path.basename(d)
            if not a.max_seconds and os.path.exists(os.path.join(a.out, f"{pid}.done")):
                print(f"{pid}: done, skipping")
                continue
            try:
                process_participant(d, chars, a.model, a.out, a.max_seconds)
            except Exception as e:                       # keep going; report at the end
                print(f"{pid}: FAILED {type(e).__name__}: {e}", flush=True)
        return

    import shutil
    import zipfile
    with zipfile.ZipFile(a.zip) as zf:
        groups = zip_participants(zf)
        pids = sorted(groups, key=key)
        if a.participants:
            pids = [p for p in pids if p in a.participants]
        if not pids:
            sys.exit(f"no P_* participant folders inside {a.zip}")
        print(f"{len(pids)} participants in the zip; {len(chars)} in characteristics file", flush=True)
        for pid in pids:
            if not a.max_seconds and os.path.exists(os.path.join(a.out, f"{pid}.done")):
                print(f"{pid}: done, skipping")
                continue
            g = groups[pid]
            pdir = os.path.join(a.tmp, pid)
            shutil.rmtree(a.tmp, ignore_errors=True)
            os.makedirs(pdir)
            try:
                mb = sum(zi.file_size for zi in g["need"]) / 1e6
                for zi in g["need"]:
                    with zf.open(zi) as fsrc, open(os.path.join(pdir, os.path.basename(zi.filename)), "wb") as fdst:
                        shutil.copyfileobj(fsrc, fdst, 16 * 1024 * 1024)
                if g["instr"]:
                    open(os.path.join(pdir, g["instr"]), "wb").close()     # empty stand-in; only the name is read
                print(f"{pid}: unpacked {len(g['need'])} files, {mb:.0f} MB", flush=True)
                process_participant(pdir, chars, a.model, a.out, a.max_seconds)
            except Exception as e:
                print(f"{pid}: FAILED {type(e).__name__}: {e}", flush=True)
            finally:
                shutil.rmtree(a.tmp, ignore_errors=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/e1b/eott_analyze.py
"""
E1b, step 2: does averaging over a dwell remove the frame-level scatter found in E1?

Inputs: the per-participant frame and Tobii CSVs written by eott_extract.py.

Protocol (fixed before seeing any EOTT result)
----------------------------------------------
* Ground truth comes from the Tobii Pro X3-120 (120 Hz), not from on-screen dots.
* v2 (21 Sep, after v1 found only 106 windows): dwell windows are consecutive
  non-overlapping windows of length L (0.5 s, 1 s) over each Tobii stream,
  kept when >= 70% of samples are valid and the gaze is stable,
  sqrt(var x + var y) <= 0.03 of the screen (primary; 0.02 and 0.05 reported
  as sensitivity). The window's true position is the mean of its Tobii samples.
* v2: the per-frame model is trained on every labelled frame of the other
  participants (v1 used only frames inside windows: 1,537 frames).
* Per-frame gaze model: the E1 "M2" feature set, gradient boosting, trained with
  10-fold GroupKFold over participants (no participant is ever in both train and
  test).
  Laptop and PC participants are pooled, with setup as a feature.
* A window's prediction from its frames, in four ways:
    single = the middle frame only;
    mean / median = mean or median of the per-frame predicted positions;
    vote = majority of the per-frame zones.
  Frames without a face are skipped. A window with fewer than
  MIN_FRAME_FRAC of the expected frames is WRONG (the system would not
  have answered).
* Accuracy = mean of per-participant accuracies, BCa 95% CI over participants.
* How much error is shared within a dwell: for each window, per-frame errors
  e_t are split into the window mean (shared) and deviations (independent).
  shared_frac = var(window means) / var(all frame errors). If this is near 1,
  averaging cannot help.
"""
from __future__ import annotations

import argparse
import glob
import json
import os
import sys

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.join(HERE, "..", "e1"))
from analyze import EYE, BS, HEAD, zone_of, boot_ci   # noqa: E402

GRIDS = [(1, 2), (2, 1), (1, 3), (2, 2), (3, 3), (4, 4)]
DWELLS_MS = [500, 1000]
DISP = 0.04            # v1 (I-DT), kept for the record
DISP_STD = [0.03, 0.02, 0.05]   # primary first, then sensitivity
MARGIN_MS = 150
MIN_FRAME_FRAC = 0.5
N_FOLDS = 10
SEED = 0
FEATS = EYE + BS + HEAD + ["is_laptop"]


def load(d: str):
    fr = sorted(glob.glob(os.path.join(d, "P_*_frames.csv")))
    tb = sorted(glob.glob(os.path.join(d, "P_*_tobii.csv")))
    if not fr:
        raise SystemExit(f"no P_*_frames.csv in {d}")
    F = pd.concat([pd.read_csv(f) for f in fr], ignore_index=True)
    T = pd.concat([pd.read_csv(f) for f in tb], ignore_index=True)
    F["detected"] = F["detected"].fillna(0).astype(int)
    F["is_laptop"] = (F["setup"].str.lower() == "laptop").astype(int)
    return F, T


def fixations(t: pd.DataFrame, min_ms: float) -> list[tuple[float, float]]:
    """Dispersion-threshold (I-DT) fixations on one task's Tobii stream.
    Returns (start_ms, end_ms) for every fixation lasting at least min_ms."""
    t = t[t.tvalid == 1].dropna(subset=["tx", "ty"])
    ts, xs, ys = t.t_ms.values, t.tx.values, t.ty.values
    n = len(ts)

    def disp(i, j):
        return (xs[i:j + 1].max() - xs[i:j + 1].min()) + (ys[i:j + 1].max() - ys[i:j + 1].min())

    out, i = [], 0
    while i < n:
        j = np.searchsorted(ts, ts[i] + min_ms)          # first sample at or beyond min_ms
        if j >= n:
            break
        if disp(i, j) <= DISP and np.diff(ts[i:j + 1]).max(initial=0) <= 50:
            while j + 1 < n and ts[j + 1] - ts[j] < 50 and disp(i, j + 1) <= DISP:
                j += 1
            out.append((float(ts[i]), float(ts[j])))
            i = j + 1
        else:
            i += 1
    return out


def make_windows(T: pd.DataFrame, L: float, disp_std: float) -> pd.DataFrame:
    """Non-overlapping windows of length L laid end to end over each video's Tobii
    stream. A window is a dwell if >= 70% of its Tobii samples are valid and the
    gaze is stable: sqrt(var x + var y) <= disp_std (normalised screen units).
    (v1 required a whole I-DT fixation of L + 300 ms; on EOTT that left only 106
    windows of 0.5 s in 51 people, too few to estimate anything.)"""
    out = []
    for (p, task, video), t in T.groupby(["participant", "task", "video"]):
        t = t.sort_values("t_ms")
        t0 = t.t_ms.min()
        k = ((t.t_ms - t0) // L).astype(int)
        v = t[t.tvalid == 1]
        g = v.groupby(k[v.index])
        st = pd.DataFrame({"n": g.size(), "gx": g.tx.mean(), "gy": g.ty.mean(),
                           "sx": g.tx.std(), "sy": g.ty.std()})
        st = st[(st.n >= 0.7 * L / 1000 * 120) & (np.sqrt(st.sx ** 2 + st.sy ** 2) <= disp_std)]
        for kk, r in st.iterrows():
            out.append({"participant": p, "task": task, "video": video, "w_start": t0 + kk * L,
                        "w_end": t0 + (kk + 1) * L, "gx": r.gx, "gy": r.gy})
    cols = ["participant", "task", "video", "w_start", "w_end", "gx", "gy"]
    W = pd.DataFrame(out, columns=cols)
    return W[np.isfinite(W.gx) & np.isfinite(W.gy)].sort_values(["participant", "video", "w_start"]).reset_index(drop=True)


def assign_frames(F: pd.DataFrame, W: pd.DataFrame) -> pd.Series:
    """window id for each frame (or -1)."""
    wid = pd.Series(-1, index=F.index)
    for (p, video), w in W.groupby(["participant", "video"]):
        m = (F.participant == p) & (F.video == video)
        e = F.loc[m, "epoch_ms"].values
        k = np.searchsorted(w.w_start.values, e, side="right") - 1
        inside = (k >= 0) & (e < w.w_end.values[np.clip(k, 0, len(w) - 1)])
        wid.loc[m] = np.where(inside, w.index.values[np.clip(k, 0, len(w) - 1)], -1)
    return wid


def fit_predict(F: pd.DataFrame, train_mask: np.ndarray) -> np.ndarray:
    pred = np.full((len(F), 2), np.nan)
    det = F.detected.values == 1
    groups = F.participant.values
    n_groups = len(np.unique(groups))
    gkf = GroupKFold(n_splits=min(N_FOLDS, n_groups))
    X = F[FEATS].values
    for k, (tr, te) in enumerate(gkf.split(X, groups=groups)):
        trm = np.zeros(len(F), bool); trm[tr] = True
        trm &= train_mask & det
        tem = np.zeros(len(F), bool); tem[te] = True
        tem &= det
        for j, tgt in enumerate(("tobii_x", "tobii_y")):
            m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                                              early_stopping=False, random_state=SEED)
            m.fit(X[trm], F[tgt].values[trm])
            pred[tem, j] = m.predict(X[tem])
        print(f"  fold {k + 1}/{gkf.get_n_splits()}", flush=True)
    return pred


def window_predictions(F, pred, W, L, fps_by_video):
    out = {}
    Fw = F.assign(px=pred[:, 0], py=pred[:, 1])
    Fw = Fw[Fw.wid >= 0]
    g = Fw.groupby("wid")
    for method in ("single", "mean", "median", "vote"):
        out[method] = {}
    expected = {i: max(1.0, fps_by_video.get(v, 30.0) * L / 1000) for i, v in zip(W.index, W.video)}
    centre = ((W.w_start + W.w_end) / 2).to_dict()
    for wid, d in g:
        d = d[d.detected == 1]
        if len(d) < MIN_FRAME_FRAC * expected[wid]:
            continue
        mid = d.iloc[int(np.argmin(np.abs(d.epoch_ms.values - centre[wid])))]
        out["single"][wid] = (mid.px, mid.py)
        out["mean"][wid] = (d.px.mean(), d.py.mean())
        out["median"][wid] = (d.px.median(), d.py.median())
        out["vote"][wid] = d[["px", "py"]].values          # zone vote done per grid
    return out


def score_windows(F, pred, W, L, fps_by_video, rng):
    """Accuracy per grid and aggregation method, plus the shared-error ICC."""
    rows, shared = [], []
    F = F.copy()
    F["wid"] = assign_frames(F, W)
    wp = window_predictions(F, pred, W, L, fps_by_video)
    for (r, c) in GRIDS:
        zt = pd.Series(zone_of(W.gx, W.gy, r, c), index=W.index)
        for method, d in wp.items():
            zp = pd.Series(-1, index=W.index)          # unanswered windows are wrong
            for wid, v in d.items():
                if method == "vote":
                    z = zone_of(v[:, 0], v[:, 1], r, c)
                    z = z[z >= 0]
                    if len(z):
                        cnt = np.bincount(z, minlength=r * c)
                        top = np.flatnonzero(cnt == cnt.max())
                        zp[wid] = top[0] if len(top) == 1 else zone_of([v[:, 0].mean()], [v[:, 1].mean()], r, c)[0]
                else:
                    zp[wid] = zone_of([v[0]], [v[1]], r, c)[0]
            ok = ((zp == zt) & (zp >= 0) & (zt >= 0)).groupby(W.participant).mean()
            lo, hi = boot_ci(ok.values, rng)
            rows.append({"grid": f"{r}x{c}", "method": method, "acc": ok.mean(), "ci_lo": lo, "ci_hi": hi,
                         "worst": ok.min(), "best": ok.max(), "answered_frac": len(d) / max(len(W), 1)})
    d = F[(F.wid >= 0) & (F.detected == 1)].join(W[["gx", "gy"]], on="wid")
    for ax, g in (("x", "gx"), ("y", "gy")):
        e = d[f"p{ax}"] - d[g]
        grp = e.groupby(d.wid)
        wm = grp.transform("mean")
        k, N = grp.ngroups, len(e)
        n_i = grp.size().values
        msb = float((n_i * (grp.mean().values - e.mean()) ** 2).sum() / max(k - 1, 1))
        msw = float(((e - wm) ** 2).sum() / max(N - k, 1))
        n0 = (N - (n_i ** 2).sum() / N) / max(k - 1, 1)
        icc = (msb - msw) / (msb + (n0 - 1) * msw) if (msb + (n0 - 1) * msw) > 0 else np.nan
        shared.append({"axis": ax, "shared_frac_icc": float(icc), "shared_frac_naive": float(wm.var() / e.var()),
                       "frames_per_window_mean": float(n_i.mean()),
                       "rmse_frame": float(np.sqrt((e ** 2).mean())),
                       "rmse_window_mean": float(np.sqrt((grp.mean() ** 2).mean()))})
    return rows, shared


def lag_check(F, T, pred):
    """Per participant: lag (ms) at which predicted x best matches Tobii x.
    Near 0 means webcam frames and Tobii samples are correctly aligned in time."""
    out = []
    Fp = F.assign(px=pred[:, 0])
    for p, d in Fp[Fp.px.notna()].groupby("participant"):
        best = (np.nan, -np.inf)
        tp = T[(T.participant == p) & (T.tvalid == 1)].dropna(subset=["tx"])
        rs = {}
        for v, dv in d.groupby("video"):
            tv = tp[tp.video == v].sort_values("t_ms")
            dv = dv.sort_values("epoch_ms")
            if len(tv) < 100 or len(dv) < 100:
                continue
            for lag in range(-600, 601, 33):
                j = np.interp(dv.epoch_ms.values + lag, tv.t_ms.values, tv.tx.values)
                rs.setdefault(lag, []).append(np.corrcoef(dv.px.values, j)[0, 1])
        if rs:
            lag, r = max(((l, float(np.nanmean(v))) for l, v in rs.items()), key=lambda z: z[1])
            r0 = float(np.nanmean(rs[min(rs, key=abs)]))
            out.append({"participant": p, "best_lag_ms": lag, "r_at_best": r, "r_at_zero": r0})
    return pd.DataFrame(out)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--features", default="eott_features")
    ap.add_argument("--out", default="e1b_results")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    rng = np.random.default_rng(SEED)

    F, T = load(a.features)
    parts = sorted(F.participant.unique(), key=lambda s: int(s.split("_")[1]))
    qa = {"version": "v2 (stability windows; model trained on all labelled frames)",
          "n_participants": len(parts),
          "setups": F.groupby("setup").participant.nunique().to_dict(),
          "n_frames": int(len(F)),
          "face_detected": float(F.detected.mean()),
          "frames_with_tobii_label": float(F.tobii_x.notna().mean()),
          "median_tobii_dt_ms": float(F.tobii_dt_ms.median())}
    fps_by_video = {}
    for v, d in F.groupby(["participant", "video"]):
        span = (d.t_rel_ms.max() - d.t_rel_ms.min()) / 1000
        fps_by_video[v[1]] = (len(d) - 1) / span if span > 0 else 30.0
    qa["median_fps"] = float(np.median(list(fps_by_video.values())))

    # per-frame model on every frame with a face and a Tobii label (other participants only)
    train_mask = F.tobii_x.notna().values & F.tobii_y.notna().values
    qa["train_frames"] = int((train_mask & (F.detected.values == 1)).sum())
    print(json.dumps(qa, indent=2), flush=True)
    print("fitting per-frame model ...", flush=True)
    pred = fit_predict(F, train_mask)
    F["px"], F["py"] = pred[:, 0], pred[:, 1]

    lag = lag_check(F, T, pred)
    lag.to_csv(os.path.join(a.out, "e1b_lag_check.csv"), index=False)
    qa["lag_median_ms"] = float(lag.best_lag_ms.median())
    qa["lag_within_200ms_frac"] = float((lag.best_lag_ms.abs() <= 200).mean())
    qa["r_predx_tobiix_at_zero_median"] = float(lag.r_at_zero.median())

    # frame-level accuracy on all labelled frames (comparable with E1)
    fl = F[train_mask]
    frame_rows = []
    for (r, c) in GRIDS:
        ok = pd.Series(zone_of(fl.px, fl.py, r, c) == zone_of(fl.tobii_x, fl.tobii_y, r, c), index=fl.index)
        ps = ok.groupby(fl.participant).mean()
        lo, hi = boot_ci(ps.values, rng)
        frame_rows.append({"grid": f"{r}x{c}", "acc": ps.mean(), "ci_lo": lo, "ci_hi": hi})
    frame_acc = pd.DataFrame(frame_rows)

    rows, shared, wcount = [], [], []
    for disp in DISP_STD:
        for L in DWELLS_MS:
            W = make_windows(T, L, disp)
            npart = W.participant.nunique()
            wcount.append({"disp_std": disp, "dwell_ms": L, "windows": len(W), "participants": npart,
                           "median_per_participant": float(W.groupby("participant").size().median()) if len(W) else 0})
            print(f"disp {disp} dwell {L} ms: {len(W)} windows from {npart} participants", flush=True)
            if len(W) == 0:
                continue
            r_, s_ = score_windows(F, pred, W, L, fps_by_video, rng)
            for x in r_: x.update(disp_std=disp, dwell_ms=L)
            for x in s_: x.update(disp_std=disp, dwell_ms=L)
            rows += r_; shared += s_
    acc, sh, wc = pd.DataFrame(rows), pd.DataFrame(shared), pd.DataFrame(wcount)
    acc.to_csv(os.path.join(a.out, "e1b_dwell_accuracy.csv"), index=False)
    frame_acc.to_csv(os.path.join(a.out, "e1b_frame_accuracy.csv"), index=False)
    sh.to_csv(os.path.join(a.out, "e1b_shared_error.csv"), index=False)
    wc.to_csv(os.path.join(a.out, "e1b_window_counts.csv"), index=False)
    with open(os.path.join(a.out, "e1b_qa.json"), "w") as f:
        json.dump(qa, f, indent=2)

    P = DISP_STD[0]
    L_ = ["# E1b (v2): dwell-level gaze-zone accuracy on EOTT (webcam video, Tobii ground truth)", "",
          f"{qa['n_participants']} participants ({qa['setups']}), {qa['n_frames']} frames, "
          f"face detected in {qa['face_detected']:.1%}, median {qa['median_fps']:.1f} fps; "
          f"per-frame model trained on {qa['train_frames']} labelled frames (10-fold, grouped by participant).", "",
          f"**Timing check:** best lag between predicted and Tobii x has median {qa['lag_median_ms']:.0f} ms; "
          f"{qa['lag_within_200ms_frac']:.0%} of participants within ±200 ms.", "",
          "## Windows found", "", "| stability (std) | dwell | windows | participants | median per participant |",
          "|---|---|---|---|---|"]
    for _, x in wc.iterrows():
        L_.append(f"| {x.disp_std} | {int(x.dwell_ms)} ms | {int(x.windows)} | {int(x.participants)} | {x.median_per_participant:.0f} |")
    L_ += ["", "## Single frames (all labelled frames; comparable with E1)", "", "| Grid | Accuracy | 95% CI |", "|---|---|---|"]
    for _, x in frame_acc.iterrows():
        L_.append(f"| {x.grid} | {x.acc:.3f} | {x.ci_lo:.3f}–{x.ci_hi:.3f} |")
    for L in DWELLS_MS:
        s = acc[(acc.disp_std == P) & (acc.dwell_ms == L)] if len(acc) else acc
        if not len(s):
            continue
        L_ += ["", f"## Dwell {L} ms (primary stability threshold {P})", "",
               "| Grid | single frame | mean | median | vote | answered |", "|---|---|---|---|---|---|"]
        for g in [f"{r}x{c}" for r, c in GRIDS]:
            q = s[s.grid == g].set_index("method")
            L_.append(f"| {g} | {q.loc['single','acc']:.3f} | {q.loc['mean','acc']:.3f} "
                      f"({q.loc['mean','ci_lo']:.3f}–{q.loc['mean','ci_hi']:.3f}) | {q.loc['median','acc']:.3f} | "
                      f"{q.loc['vote','acc']:.3f} | {q.loc['mean','answered_frac']:.2f} |")
    L_ += ["", "## Sensitivity: mean-of-dwell accuracy at other stability thresholds", "",
           "| stability | dwell | 1x2 | 1x3 | 2x2 | 3x3 |", "|---|---|---|---|---|---|"]
    for (d_, L), s in acc[acc.method == "mean"].groupby(["disp_std", "dwell_ms"]) if len(acc) else []:
        g = s.set_index("grid").acc
        L_.append(f"| {d_} | {L} ms | {g['1x2']:.3f} | {g['1x3']:.3f} | {g['2x2']:.3f} | {g['3x3']:.3f} |")
    L_ += ["", "## How much of the error is shared across a dwell", "",
           "ICC near 1: averaging cannot help. Near 0: averaging removes most of the error.", "",
           "| stability | dwell | axis | ICC | RMSE single frame | RMSE dwell mean | frames per window |",
           "|---|---|---|---|---|---|---|"]
    for _, x in sh.iterrows():
        L_.append(f"| {x.disp_std} | {x.dwell_ms} ms | {x.axis} | {x.shared_frac_icc:.2f} | {x.rmse_frame:.3f} | "
                  f"{x.rmse_window_mean:.3f} | {x.frames_per_window_mean:.1f} |")
    L_ += ["", "## QA", "", "```", json.dumps(qa, indent=2), "```"]
    with open(os.path.join(a.out, "E1b_REPORT.md"), "w") as f:
        f.write("\n".join(L_) + "\n")
    print("\n".join(L_))


if __name__ == "__main__":
    main()


## 2. Dry run: first 40 s of each task for one participant
Every line must start with **OK** or **INFO**. If any says **FAIL**, stop and read this cell's output before going on.

In [ ]:
P1 = sorted(parts, key=lambda p: int(p.split('_')[1]))[0]; print('dry run on', P1)
!cd /content/e1b && python eott_extract.py --zip /content/eott.zip --chars /content/participant_characteristics.csv --model /content/face_landmarker.task --out /content/e1b_dry --participants $P1 --max-seconds 40
import pandas as pd, numpy as np
d = pd.read_csv(f'/content/e1b_dry/{P1}_frames.dry.csv')
lab = d.dropna(subset=['tobii_x'])
# timing check, per video: the iris feature should track Tobii x best at a lag near 0
tb = pd.read_csv(f'/content/e1b_dry/{P1}_tobii.dry.csv').dropna(subset=['tx']).sort_values('t_ms')
best = None
if 'reye_h' in d and d.reye_h.notna().sum() > 50:
    res = {}
    for v, dv in d.dropna(subset=['reye_h']).sort_values('epoch_ms').groupby('video'):
        tv = tb[tb.video == v]
        if len(tv) < 50 or len(dv) < 50: continue
        for lag in range(-1000, 1001, 33):
            j = np.interp(dv.epoch_ms.values + lag, tv.t_ms.values, tv.tx.values)
            res.setdefault(lag, []).append(np.corrcoef(dv.reye_h.values, j)[0, 1])
    if res:
        best = max(((lag, float(np.nanmean(r))) for lag, r in res.items()), key=lambda z: abs(z[1]))
checks = {
  'tasks found': ', '.join(sorted(d.task.unique())),
  'frames': str(len(d)),
  'frame size': f"{int(d.img_w.iloc[0])}x{int(d.img_h.iloc[0])}",
  'setup': d.setup.iloc[0],
  'best lag (ms) and correlation, iris vs Tobii x': str(best),
  'face detected in >= 85% of frames': d.detected.mean() >= 0.85,
  'Tobii label on >= 60% of frames': d.tobii_x.notna().mean() >= 0.60,
  'Tobii gaze inside the screen (median x, y in 0-1)': lab.tobii_x.between(0,1).mean() > 0.9 and lab.tobii_y.between(0,1).mean() > 0.9,
  'frame clock and Tobii clock agree within 0.5 s (best lag near 0)': best is not None and abs(best[0]) <= 500 and abs(best[1]) >= 0.10,
}
for k, v in checks.items():
    if isinstance(v, str): print('INFO ', k, ':', v)
    else: print(('OK   ' if bool(v) else 'FAIL ') + k)
print('face detected', round(d.detected.mean(), 3), '| Tobii-labelled', round(d.tobii_x.notna().mean(), 3))

## 3. Full extraction, all participants (long: roughly 1–3 h; resumable; saved to Drive). Participants are unpacked one at a time and deleted, so the disk never fills.

In [ ]:
!cd /content/e1b && python eott_extract.py --zip /content/eott.zip --chars /content/participant_characteristics.csv --model /content/face_landmarker.task --out "{WORK}/features"

## 4. Analysis (~10–20 min)

In [ ]:
!cd /content/e1b && python eott_analyze.py --features "{WORK}/features" --out "{WORK}/results"

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{WORK}/results/E1b_REPORT.md').read()))

## 5. Download the results (small; the features stay on your Drive)

In [ ]:
import shutil
shutil.make_archive('/content/E1b_results', 'zip', f'{WORK}/results')
from google.colab import files; files.download('/content/E1b_results.zip')